# D02 — External Biological Databases

Downloads and caches the external reference databases used for biological
annotation and external validation of outlier genes.

**Outputs (in `data/external/`):**
- `gnomad.v4.1.constraint_metrics.tsv` — gnomAD v4.1 gene constraint (~92 MB)
- `CRISPRGeneEffect.csv` — DepMap 25Q3 CRISPR dependency (~413 MB)
- `clinvar_gene_specific_summary.txt` — ClinVar gene-level pathogenicity (~3.4 MB)

**Outputs (in `data/`):**
- `hpa_rna_consensus.tsv.zip` — Human Protein Atlas tissue expression (~5 MB)

**Prerequisites:** `requests`, `pandas`

## Section 1: gnomAD v4.1 Constraint Metrics

gnomAD provides population-level constraint metrics. pLI (probability of loss-of-function intolerance) and LOEUF (loss-of-function observed/expected upper fraction) quantify selective constraint against gene inactivation. Highly constrained genes (pLI > 0.9) are more likely to be essential.

In [1]:
import requests
from pathlib import Path
import os

DATA_DIR = Path('data/external')
DATA_DIR.mkdir(parents=True, exist_ok=True)

GNOMAD_URL = 'https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/constraint/gnomad.v4.1.constraint_metrics.tsv'
GNOMAD_PATH = DATA_DIR / 'gnomad.v4.1.constraint_metrics.tsv'

if GNOMAD_PATH.exists():
    size_mb = GNOMAD_PATH.stat().st_size / 1e6
    print(f'gnomAD constraint metrics already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading gnomAD v4.1 constraint metrics...')
    print(f'  Source: {GNOMAD_URL}')
    resp = requests.get(GNOMAD_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(GNOMAD_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {GNOMAD_PATH}')

gnomAD constraint metrics already cached (95.5 MB)


In [2]:
import pandas as pd

gnomad = pd.read_csv(GNOMAD_PATH, sep='\t', nrows=5)
print(f'Columns ({len(gnomad.columns)}): {list(gnomad.columns[:10])}...')

# Column names vary by gnomAD version — find the right ones
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

gnomad_full = pd.read_csv(GNOMAD_PATH, sep='\t', low_memory=False)
gene_col  = find_col(gnomad_full, ['gene', 'gene_id', 'symbol'])
pli_col   = find_col(gnomad_full, ['lof.pLI', 'pLI'])
loeuf_col = find_col(gnomad_full, ['lof.oe_ci.upper', 'oe_lof_upper', 'LOEUF'])

print(f'Loaded {len(gnomad_full):,} rows')
print(f'Gene column: {gene_col} ({gnomad_full[gene_col].nunique():,} unique genes)')
print(f'pLI column: {pli_col}')
print(f'LOEUF column: {loeuf_col}')


Columns (55): ['gene', 'gene_id', 'transcript', 'canonical', 'mane_select', 'lof_hc_lc.obs', 'lof_hc_lc.exp', 'lof_hc_lc.possible', 'lof_hc_lc.oe', 'lof_hc_lc.mu']...


Loaded 211,523 rows
Gene column: gene (18,203 unique genes)
pLI column: lof.pLI
LOEUF column: lof.oe_ci.upper


## Section 2: DepMap CRISPR Gene Effect (25Q3)

The Cancer Dependency Map (DepMap) provides genome-wide CRISPR knockout screens. The CRISPRGeneEffect.csv contains gene dependency scores (Chronos algorithm) where more negative values indicate stronger growth inhibition upon gene knockout.

In [3]:
import csv
import io

DEPMAP_RELEASE = 'DepMap Public 25Q3'
DEPMAP_FILENAME = 'CRISPRGeneEffect.csv'
DEPMAP_PATH = DATA_DIR / 'CRISPRGeneEffect.csv'

# DepMap now uses time-limited signed Google Cloud Storage URLs.
# The file-listing API provides fresh signed URLs, but some networks /
# user-agents get blocked.  We try the API first with a browser-like
# user-agent, then fall back to the direct (unsigned) GCS URL.
_DEPMAP_SESSION = requests.Session()
_DEPMAP_SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (compatible; glitch-genes-pipeline/1.0)'
})

def get_depmap_download_url(release, filename):
    """Fetch a fresh signed download URL from the DepMap file-listing API."""
    api_url = 'https://depmap.org/portal/api/download/files'
    print(f'  Querying DepMap download API for {release} / {filename} ...')
    try:
        resp = _DEPMAP_SESSION.get(api_url, timeout=30)
        resp.raise_for_status()
        reader = csv.DictReader(io.StringIO(resp.text))
        for row in reader:
            if row['release'] == release and row['filename'] == filename:
                print(f'  Found signed URL via API')
                return row['url']
        # Release not found in listing
        resp2 = _DEPMAP_SESSION.get(api_url, timeout=30)
        reader2 = csv.DictReader(io.StringIO(resp2.text))
        available = sorted({r['release'] for r in reader2
                           if filename in r.get('filename', '')})
        raise FileNotFoundError(
            f'{filename} not found in release "{release}". '
            f'Available releases: {available}'
        )
    except requests.exceptions.RequestException as e:
        print(f'  API request failed ({e}) — falling back to direct GCS URL')
        # Direct unsigned GCS URL for 25Q3 (canonical path is stable)
        gcs_path = 'downloads-by-canonical-id/25q3-public-6202.1/CRISPRGeneEffect.csv'
        return f'https://storage.googleapis.com/depmap-external-downloads/{gcs_path}'

if DEPMAP_PATH.exists():
    size_mb = DEPMAP_PATH.stat().st_size / 1e6
    print(f'DepMap CRISPR gene effect already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading {DEPMAP_RELEASE} CRISPR gene effect...')
    depmap_url = get_depmap_download_url(DEPMAP_RELEASE, DEPMAP_FILENAME)
    resp = _DEPMAP_SESSION.get(depmap_url, stream=True, timeout=300)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(DEPMAP_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {DEPMAP_PATH}')


DepMap CRISPR gene effect already cached (432.3 MB)


In [4]:
depmap = pd.read_csv(DEPMAP_PATH, nrows=1)
print(f'DepMap CRISPR gene effect: {depmap.shape[0]} rows (loaded sample), {depmap.shape[1]} cell lines (genes)')
print(f'First 10 columns: {list(depmap.columns[:10])}')

DepMap CRISPR gene effect: 1 rows (loaded sample), 18436 cell lines (genes)
First 10 columns: ['Unnamed: 0', 'A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)', 'A3GALT2 (127550)', 'A4GALT (53947)', 'A4GNT (51146)', 'AAAS (8086)', 'AACS (65985)']


## Section 3: ClinVar Gene-Specific Summary

ClinVar aggregates genetic variant interpretations. The gene_specific_summary file lists per-gene counts of pathogenic, likely pathogenic, and benign variants, enabling classification of genes as "disease-associated."

In [5]:
CLINVAR_URL = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/gene_specific_summary.txt'
CLINVAR_PATH = DATA_DIR / 'clinvar_gene_specific_summary.txt'

if CLINVAR_PATH.exists():
    size_mb = CLINVAR_PATH.stat().st_size / 1e6
    print(f'ClinVar gene-specific summary already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading ClinVar gene-specific summary...')
    print(f'  Source: {CLINVAR_URL}')
    resp = requests.get(CLINVAR_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(CLINVAR_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {CLINVAR_PATH}')

ClinVar gene-specific summary already cached (3.5 MB)


In [6]:
# ClinVar gene-specific summary has a comment header line starting with '#'.
# Read first line to decide whether to skip it (robust to format changes).
with open(CLINVAR_PATH) as _f:
    _first = _f.readline()
_skip = 1 if _first.startswith('#') else 0
clinvar = pd.read_csv(CLINVAR_PATH, sep='\t', skiprows=_skip, low_memory=False)
# Strip any leading '#' from column names (defensive)
clinvar.columns = [c.lstrip('#').strip() for c in clinvar.columns]
print(f'ClinVar gene-specific summary: {clinvar.shape}')
print(f'Columns: {list(clinvar.columns)}')

ClinVar gene-specific summary: (92700, 9)
Columns: ['Symbol', 'GeneID', 'Total_submissions', 'Total_alleles', 'Submissions_reporting_this_gene', 'Alleles_reported_Pathogenic_Likely_pathogenic', 'Gene_MIM_number', 'Number_uncertain', 'Number_with_conflicts']


## Section 4: Human Protein Atlas (HPA) RNA Consensus

HPA provides consensus tissue expression levels (nTPM) across ~60 human tissues. Used to compute expression breadth (number of tissues where a gene is expressed above 1 TPM) and maximum tissue expression.

In [7]:
import zipfile

HPA_URL = 'https://www.proteinatlas.org/download/tsv/rna_tissue_consensus.tsv.zip'
HPA_PATH = Path('data/hpa_rna_consensus.tsv.zip')
HPA_PATH.parent.mkdir(parents=True, exist_ok=True)

if HPA_PATH.exists():
    size_mb = HPA_PATH.stat().st_size / 1e6
    print(f'HPA RNA consensus already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading Human Protein Atlas RNA consensus...')
    print(f'  Source: {HPA_URL}')
    resp = requests.get(HPA_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(HPA_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {HPA_PATH}')

HPA RNA consensus already cached (5.3 MB)


In [8]:
with zipfile.ZipFile(HPA_PATH, 'r') as zf:
    file_list = zf.namelist()
    print(f'ZIP contents: {file_list}')
    with zf.open(file_list[0]) as f:
        hpa = pd.read_csv(f, sep='\t', nrows=1000)
        print(f'HPA RNA consensus: {hpa.shape}')
        print(f'First 5 columns: {list(hpa.columns[:5])}')

ZIP contents: ['rna_tissue_consensus.tsv']
HPA RNA consensus: (1000, 4)
First 5 columns: ['Gene', 'Gene name', 'Tissue', 'nTPM']


## Section 5: Verification Summary

Verify all downloaded files exist and report their sizes.

In [9]:
print('External Database Files:')
print('=' * 60)

files_to_check = [
    (GNOMAD_PATH, 'gnomAD v4.1 constraint metrics'),
    (DEPMAP_PATH, 'DepMap 25Q3 CRISPR gene effect'),
    (CLINVAR_PATH, 'ClinVar gene-specific summary'),
    (HPA_PATH, 'HPA RNA consensus (zipped)')
]

total_size = 0
for path, description in files_to_check:
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        total_size += path.stat().st_size
        status = '✓'
    else:
        size_mb = 0
        status = '✗'
    print(f'{status} {description:.<45} {size_mb:>8.1f} MB')

print('=' * 60)
print(f'Total size: {total_size / 1e9:.2f} GB')

External Database Files:
✓ gnomAD v4.1 constraint metrics...............     95.5 MB
✓ DepMap 25Q3 CRISPR gene effect...............    432.3 MB
✓ ClinVar gene-specific summary................      3.5 MB
✓ HPA RNA consensus (zipped)...................      5.3 MB
Total size: 0.54 GB
